# Chronos-2 Training & Benchmarking

This notebook establishes a training baseline for Chronos-2 using a mix of synthetic data and real-world datasets (Chronos datasets, GiftEval).

## 1. Setup & Configuration

In [2]:
# Clone Repository
!git clone https://github.com/emanueleromito/voyagers-forecasting.git
%cd voyagers-forecasting

# Create checkpoint directory
import os
CHECKPOINT_DIR = './checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

Cloning into 'voyagers-forecasting'...
remote: Enumerating objects: 1017, done.
remote: Counting objects: 100% (455/455), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 1017 (delta 357), reused 293 (delta 293), pack-reused 562 (from 2)
Receiving objects: 100% (1017/1017), 2.42 MiB | 9.13 MiB/s, done.
Resolving deltas: 100% (501/501), done.
/content/voyagers-forecasting/voyagers-forecasting


In [3]:
# Install dependencies
!pip install -e .[dev]
!pip install gluonts transformers accelerate typer typer-config rich wandb datasets

# Fix SymPy compatibility issue with PyTorch
!pip install --upgrade sympy

Obtaining file:///content/voyagers-forecasting/voyagers-forecasting
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver w

In [4]:
import sys
import os
import random
import torch
import numpy as np
import wandb
import transformers
import datasets
import math
from pathlib import Path
import pandas as pd
from typing import Optional, List, Iterator, Sequence, Mapping, Any, Union
from huggingface_hub import hf_hub_download
from huggingface_hub import HfApi
from transformers import TrainingArguments
from google.colab import userdata
from gluonts.dataset.common import FileDataset
from torch.utils.data import IterableDataset

sys.path.append(os.path.abspath("src"))

from scripts.evaluation.evaluate import eval_pipeline_and_save_results
from chronos2.pipeline import Chronos2Pipeline
from chronos2.config import Chronos2CoreConfig, Chronos2ForecastingConfig
from chronos2.model import Chronos2Model
from chronos2.dataset import Chronos2Dataset, DatasetMode, left_pad_and_cat_2D, validate_and_prepare_single_dict_task
from chronos2.trainer import Chronos2Trainer

## 1.1 Configuration & Hyperparameters

In [5]:
# --- Secrets ---
HF_TOKEN = userdata.get('HF_TOKEN')
WANDB = userdata.get('wandb')

# --- Reproducibility ---
SEED = 42

# --- Model Configuration ---
CONTEXT_LENGTH = 8192   # Chronos-2 supports up to 8192 context steps
PREDICTION_LENGTH = 1024 # Max prediction horizon (64 patches * 16 steps)
PATCH_SIZE = 16
D_MODEL = 768           # 120M model uses 768
D_FF = 3072             # Feed-forward dimension (typically 4 * d_model)
D_KV = 64               # Key/Value dimension (d_model / num_heads = 768 / 12)
NUM_LAYERS = 12         # 12 layers for the 120M parameter model
NUM_HEADS = 12          # 12 attention heads
DROPOUT_RATE = 0.1      # Standard dropout rate
VOCAB_SIZE = 2
QUANTILES = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]

# --- Training Configuration ---
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_STEPS = 5000
SAVE_STEPS = 1000
LOGGING_STEPS = 100
WARMUP_RATIO = 0.0
RUN_NAME = "chronos2-baseline"

TimeoutException: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# --- Reproducibility Setup ---
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    
    print(f"Random seed set to {seed}")

set_seed(SEED)

# Check for GPU
if torch.cuda.is_available():
    print(f"\nUsing GPU: {torch.cuda.get_device_name(0)}")
else:
    print("\nWARNING: GPU not available. Training will be slow.")

In [ ]:
wandb.login(key=WANDB)

## 2. Dataset Implementation (Streaming)

We implement a custom `StreamingChronosDataset` that handles mixing multiple streaming datasets and yielding batches directly, as required by `Chronos2Trainer`.

In [ ]:
class StreamingChronosDataset(IterableDataset):
    def __init__(
        self,
        datasets: List[Any],
        probabilities: List[float],
        context_length: int,
        prediction_length: int,
        batch_size: int,
        output_patch_size: int,
        min_past: int = 1,
        mode: str = DatasetMode.TRAIN,
    ):
        super().__init__()
        assert len(datasets) == len(probabilities), f"Number of datasets ({len(datasets)}) must match number of probabilities ({len(probabilities)})"
        self.datasets = datasets
        self.probabilities = probabilities
        self.context_length = context_length
        self.prediction_length = prediction_length
        self.batch_size = batch_size
        self.output_patch_size = output_patch_size
        self.min_past = min_past
        self.mode = mode
        self.max_output_patches = math.ceil(prediction_length / output_patch_size)

    def _get_stream_iterators(self):
        return [iter(ds) for ds in self.datasets]

    def _construct_slice(self, task):
        # task is a tuple returned by validate_and_prepare_single_dict_task
        (
            task_past_tensor,
            task_future_tensor,
            task_n_targets,
            task_n_covariates,
            task_n_future_covariates,
        ) = task
        
        # Clone to avoid side effects if reused (though in streaming usually not reused)
        task_past_tensor, task_future_tensor = task_past_tensor.clone(), task_future_tensor.clone()
        task_n_past_only_covariates = task_n_covariates - task_n_future_covariates
        full_length = task_past_tensor.shape[-1]

        if self.mode == DatasetMode.TRAIN:
            # slice a random subsequence
            # Ensure we have enough history
            if full_length < self.min_past + self.prediction_length:
                 # This should have been filtered, but double check
                 return None
            slice_idx = np.random.randint(self.min_past, full_length - self.prediction_length + 1)
        elif self.mode == DatasetMode.VALIDATION:
            slice_idx = full_length - self.prediction_length
        else:
            slice_idx = full_length

        if slice_idx >= self.context_length:
            task_context = task_past_tensor[:, slice_idx - self.context_length : slice_idx]
        else:
            task_context = task_past_tensor[:, :slice_idx]

        if self.mode in [DatasetMode.TRAIN, DatasetMode.VALIDATION]:
            task_future_target = task_past_tensor[:, slice_idx : slice_idx + self.prediction_length].clone()
            task_future_target[task_n_targets:] = torch.nan

            if task_n_future_covariates > 0:
                task_future_covariates = task_past_tensor[
                    -task_n_future_covariates:, slice_idx : slice_idx + self.prediction_length
                ]
            else:
                task_future_covariates = torch.zeros((0, self.prediction_length))

            task_future_covariates_padding = torch.full(
                (task_n_targets + task_n_past_only_covariates, self.prediction_length),
                fill_value=torch.nan,
            )
            task_future_covariates = torch.cat([task_future_covariates_padding, task_future_covariates], dim=0)
        else:
            task_future_target = None
            task_future_covariates = task_future_tensor

        return task_context, task_future_target, task_future_covariates, task_n_targets

    def _build_batch(self, batch_samples):
        batch_context_tensor_list = []
        batch_future_target_tensor_list = []
        batch_future_covariates_tensor_list = []
        batch_group_ids_list = []
        target_idx_ranges = []

        target_start_idx = 0
        for group_id, sample in enumerate(batch_samples):
            task_context, task_future_target, task_future_covariates, task_n_targets = sample

            group_size = task_context.shape[0]
            task_group_ids = torch.full((group_size,), fill_value=group_id)
            batch_context_tensor_list.append(task_context)
            batch_future_target_tensor_list.append(task_future_target)
            batch_future_covariates_tensor_list.append(task_future_covariates)
            batch_group_ids_list.append(task_group_ids)
            target_idx_ranges.append((target_start_idx, target_start_idx + task_n_targets))
            target_start_idx += group_size

        if self.mode == DatasetMode.TRAIN:
            num_output_patches = np.random.randint(1, self.max_output_patches + 1)
        else:
            num_output_patches = self.max_output_patches
            
        horizon = num_output_patches * self.output_patch_size

        future_target = None
        if self.mode != DatasetMode.TEST:
            future_target = torch.cat(batch_future_target_tensor_list, dim=0)
            if future_target.shape[-1] > horizon:
                future_target = future_target[..., :horizon]

        future_covariates = torch.cat(batch_future_covariates_tensor_list, dim=0)
        if future_covariates.shape[-1] > horizon:
            future_covariates = future_covariates[..., :horizon]

        return {
            "context": left_pad_and_cat_2D(batch_context_tensor_list),
            "future_target": future_target,
            "future_covariates": future_covariates,
            "group_ids": torch.cat(batch_group_ids_list, dim=0),
            "num_output_patches": num_output_patches,
            "target_idx_ranges": target_idx_ranges,
        }

    def __iter__(self):
        iterators = self._get_stream_iterators()
        batch_samples = []
        current_batch_size = 0

        while True:
            # Sample a dataset
            idx = np.random.choice(len(self.datasets), p=self.probabilities)
            try:
                raw_entry = next(iterators[idx])
            except StopIteration:
                # Restart iterator if exhausted (cyclic)
                iterators[idx] = iter(self.datasets[idx])
                try:
                    raw_entry = next(iterators[idx])
                except StopIteration:
                    # Dataset is empty or cannot be restarted, skip it
                    continue

            # Prepare task
            # We need to adapt raw_entry to what validate_and_prepare_single_dict_task expects
            # It expects dict with 'target' (numpy/tensor), 'past_covariates', etc.
            # Our adapter should have already ensured this format.
            
            try:
                # Validate and prepare
                # We pass idx=0 as it's single task processing
                task = validate_and_prepare_single_dict_task(raw_entry, idx=0, prediction_length=self.prediction_length)
                
                # Filter short series
                if self.mode != DatasetMode.TEST and task[0].shape[-1] < self.min_past + self.prediction_length:
                    continue

                # Construct slice
                sample = self._construct_slice(task)
                if sample is None:
                    continue

                batch_samples.append(sample)
                current_batch_size += sample[0].shape[0] # Add group size (number of series in task)

                if current_batch_size >= self.batch_size:
                    batch = self._build_batch(batch_samples)
                    
                    # Remove target_idx_ranges for training/validation as model.forward doesn't accept it
                    if self.mode in [DatasetMode.TRAIN, DatasetMode.VALIDATION]:
                        batch.pop("target_idx_ranges")
                        
                    yield batch
                    batch_samples = []
                    current_batch_size = 0

            except Exception as e:
                # Skip bad samples
                # print(f"Error processing sample: {e}")
                continue

## 2.1 Text encoding

In [ ]:
"""
Unified text encoder for dataset descriptions.
Supports:
  - DistilBERT (mean-pooled last hidden state)
  - Sentence-BERT (sentence-transformers)

Install:
  pip install -U transformers torch sentence-transformers numpy
"""

from typing import Sequence, Optional, Literal
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel


class DatasetDescriptionEncoder:
    def __init__(
        self,
        backend: Literal["distilbert", "sbert"] = "distilbert",
        model_name: Optional[str] = None,
        max_length: int = 256,
        batch_size: int = 32,
        normalize: bool = True,
        device: Optional[str] = None,
    ):
        """
        backend:
          - "distilbert": HF Transformers + mean pooling
          - "sbert": sentence-transformers

        model_name:
          - distilbert default: "distilbert-base-uncased"
          - sbert default: "sentence-transformers/all-MiniLM-L6-v2"
        """
        self.backend = backend
        self.max_length = max_length
        self.batch_size = batch_size
        self.normalize = normalize
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        if backend == "distilbert":
            self.model_name = model_name or "distilbert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModel.from_pretrained(self.model_name).to(self.device).eval()

        elif backend == "sbert":
            self.model_name = model_name or "sentence-transformers/all-MiniLM-L6-v2"
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(self.model_name, device=self.device)

        else:
            raise ValueError(f"Unknown backend: {backend}")

    @torch.no_grad()
    def encode(self, texts: Sequence[str]) -> np.ndarray:
        """
        Returns:
          embeddings: (N, D) float32
        """
        if self.backend == "distilbert":
            return self._encode_distilbert(texts)
        else:  # sbert
            return self._encode_sbert(texts)

    def _encode_distilbert(self, texts: Sequence[str]) -> np.ndarray:
        all_embs = []

        for i in range(0, len(texts), self.batch_size):
            batch = list(texts[i : i + self.batch_size])

            enc = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            ).to(self.device)

            out = self.model(**enc)  # (B, T, H)
            hidden = out.last_hidden_state

            # Mean pooling with attention mask
            mask = enc["attention_mask"].unsqueeze(-1).type_as(hidden)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

            if self.normalize:
                pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)

            all_embs.append(pooled.cpu().numpy().astype(np.float32))

        return np.vstack(all_embs)

    def _encode_sbert(self, texts: Sequence[str]) -> np.ndarray:
        embs = self.model.encode(
            list(texts),
            batch_size=self.batch_size,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize,
            show_progress_bar=True,
        )
        return embs.astype(np.float32)


# ----------------------------
# Usage Example
# ----------------------------
if __name__ == "__main__":
    descriptions = [
        "Hourly electricity consumption with weather covariates.",
        "Daily retail sales by SKU and store.",
    ]

    enc1 = DatasetDescriptionEncoder(backend="distilbert")
    z1 = enc1.encode(descriptions)
    print("DistilBERT:", z1.shape)

    enc2 = DatasetDescriptionEncoder(backend="sbert")
    z2 = enc2.encode(descriptions)
    print("Sentence-BERT:", z2.shape)


DistilBERT: (2, 768)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Sentence-BERT: (2, 384)


## 3. Data Loading & Adapters

We load all datasets listed in the Chronos-2 paper and adapt them to the required format.

In [ ]:
# Adapter for HF Datasets
class HFDatasetAdapter:
    def __init__(self, hf_dataset, target_column=None, start_column="start", name="dataset"):
        self.hf_dataset = hf_dataset
        self.target_column = target_column
        self.start_column = start_column
        self.name = name
        self.detected_target = False

    def _detect_target_column(self, entry):
        if self.target_column:
            return self.target_column
        
        if "target" in entry:
            return "target"
            
        # Common target names
        common_names = ["consumption_kW", "power_mw", "TMAX", "t_mean", "value", "count"]
        for name in common_names:
            if name in entry:
                return name
                
        # Heuristic: First numerical column that is not ID or timestamp
        exclude = ["id", "timestamp", "start", "lat", "lng", "latitude", "longitude", "subset", "item_id", "freq", "page_name", "state", "coop_id"]
        for key, value in entry.items():
            if key not in exclude:
                # We assume it's the target if it's not excluded
                return key
                
        raise ValueError(f"Could not detect target column for {self.name}. Available keys: {list(entry.keys())}")

    def __iter__(self):
        iterator = iter(self.hf_dataset)
        try:
            first_entry = next(iterator)
        except StopIteration:
            return

        if not self.detected_target:
            self.target_column = self._detect_target_column(first_entry)
            print(f"[{self.name}] Auto-detected target column: '{self.target_column}'")
            self.detected_target = True

        # Yield first entry
        target = np.array(first_entry[self.target_column], dtype=np.float32)
        yield {"target": target}

        # Yield rest
        for entry in iterator:
            target = np.array(entry[self.target_column], dtype=np.float32)
            yield {"target": target}

# Known mappings
DATASET_TARGET_MAP = {
    "electricity_15min": "consumption_kW",
    "solar": "power_mw",
    "solar_1h": "power_mw",
    "ushcn_daily": "TMAX",
    "monash_temperature_rain": "t_mean",
}

# Load Synthetic Dataset
print("Downloading synthetic dataset...")
HF_REPO_ID = "voyagersnlppolito/model-data"
dataset_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename="synthetic_dataset.arrow",
    repo_type="dataset",
    token=HF_TOKEN
)
# Load via datasets library to allow splitting
synthetic_hf = datasets.load_dataset("arrow", data_files=str(dataset_path), split="train")
synthetic_split = synthetic_hf.train_test_split(test_size=0.1, seed=SEED)
synthetic_train = synthetic_split["train"]
synthetic_val = synthetic_split["test"]

# List of Chronos-2 Datasets (Autogluon)
chronos_dataset_names = [
    #"electricity_15min",
    "monash_australian_electricity",
    "monash_electricity_hourly",
    "monash_electricity_weekly",

    "monash_kdd_cup_2018",

    "m4_daily",
    "m4_hourly",
    "m4_monthly",
    "m4_weekly",

    "mexico_city_bikes",

    "solar",
    "solar_1h",

    "taxi_1h",
    "taxi_30min",

    "uber_tlc_daily",
    "uber_tlc_hourly",

    "ushcn_daily",
  
    "weatherbench_daily",
    "weatherbench_hourly_temperature",
    "weatherbench_weekly",
    
    "wiki_daily_100k",

    "wind_farms_daily",
    "wind_farms_hourly",

    "monash_temperature_rain",

    "monash_london_smart_meters",

]

gifteval_dataset_names = [
    "alibaba_cluster_trace_2018",
    "azure_vm_traces_2017",
    "borg_cluster_data_2011",
    "largest_2017",
    "Q-TRAFFIC",
    "buildings_900k",
]

datasets_list = []
datasets_list.append(HFDatasetAdapter(synthetic_train, name="synthetic_train"))

print("Loading real datasets...")
for name in chronos_dataset_names:
    try:
        ds = datasets.load_dataset("autogluon/chronos_datasets", name, split="train", streaming=True)
        target_col = DATASET_TARGET_MAP.get(name)
        datasets_list.append(HFDatasetAdapter(ds, target_column=target_col, name=name))
        print(f"Loaded {name}")
    except Exception as e:
        print(f"Could not load {name}: {e}")

for name in gifteval_dataset_names:
    try:
        # GiftEval datasets are in subdirectories, load using data_files
        ds = datasets.load_dataset("Salesforce/GiftEvalPretrain", data_files=f"{name}/*.arrow", split="train", streaming=True)
        datasets_list.append(HFDatasetAdapter(ds, name=name))
        print(f"Loaded {name}")
    except Exception as e:
        print(f"Could not load {name}: {e}")

# Calculate probabilities (Uniform for now, or weighted by size if known)
# To fix the ValueError, we ensure len(probs) == len(datasets)
num_datasets = len(datasets_list)
probabilities = [1.0 / num_datasets] * num_datasets

print(f"Total datasets loaded: {num_datasets}")

### 3.1 Fetching the description

In [10]:
import warnings
import re
import datasets
from huggingface_hub import hf_hub_download


def _clean_md(md: str, max_chars: int = 600) -> str:
    md = re.sub(r"```.*?```", "", md, flags=re.DOTALL)   # remove code blocks
    md = re.sub(r"!\[.*?\]\(.*?\)", "", md)             # remove images
    md = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", md)         # strip links
    md = re.sub(r"\s+", " ", md).strip()                 # normalize spaces
    return md[:max_chars]


def get_dataset_description(repo_id, config=None, token=None):
    # 1) Try official builder metadata
    try:
        builder = datasets.load_dataset_builder(repo_id, name=config)
        if builder.info.description:
            return builder.info.description.strip()
    except Exception as e:
        warnings.warn(f"[builder] failed: {e}")

    # 2) Fallback to dataset card (README.md)
    try:
        readme_path = hf_hub_download(
            repo_id=repo_id,
            filename="README.md",
            repo_type="dataset",
            token=token,
        )
        with open(readme_path, "r", encoding="utf-8") as f:
            return _clean_md(f.read())
    except Exception as e:
        warnings.warn(f"[README] failed: {e}")

    # 3) Nothing worked
    warnings.warn(f"No description found for {repo_id}/{config}")
    return ""
repo_id = "autogluon/chronos_datasets"
dataset_name = "m4_daily"

description = get_dataset_description(
    repo_id=repo_id,
    config=dataset_name,
)

print("Dataset:", dataset_name)
print("Description:")
print(description)


README.md: 0.00B [00:00, ?B/s]

Dataset: m4_daily
Description:
--- annotations_creators: - no-annotation license: other source_datasets: - original task_categories: - time-series-forecasting task_ids: - univariate-time-series-forecasting - multivariate-time-series-forecasting pretty_name: Chronos datasets dataset_info: - config_name: dominick features: - name: id dtype: string - name: timestamp sequence: timestamp[ms] - name: target sequence: float64 - name: im_0 dtype: int64 splits: - name: train num_bytes: 477140250 num_examples: 100014 download_size: 42290010 dataset_size: 477140250 - config_name: electricity_15min features: - name: id dtype: string - 


## 4. Training Execution

In [ ]:
# Create Streaming Datasets
train_ds = StreamingChronosDataset(
    datasets=datasets_list,
    probabilities=probabilities,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    batch_size=BATCH_SIZE,
    output_patch_size=PATCH_SIZE,
    mode=DatasetMode.TRAIN,
)

# Validation (use synthetic only for speed/stability)
val_ds = StreamingChronosDataset(
    datasets=[HFDatasetAdapter(synthetic_val, name="synthetic_val")],
    probabilities=[1.0],
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    batch_size=BATCH_SIZE,
    output_patch_size=PATCH_SIZE,
    mode=DatasetMode.VALIDATION,
)

# Training Arguments
training_args = TrainingArguments(
    hub_model_id=f"voyagersnlppolito/{RUN_NAME}",
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="linear",
    warmup_ratio=WARMUP_RATIO,
    max_steps=MAX_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    fp16=False,
    dataloader_num_workers=0, # Must be 0 for IterableDataset with streaming usually, or handle carefully
    dataloader_pin_memory=False, # Avoid warning if no GPU
    remove_unused_columns=False,
    report_to="wandb",
    run_name=RUN_NAME,
    seed=SEED,
    data_seed=SEED,
)

# Initialize Model (Same as before)
chronos_forecasting_config = Chronos2ForecastingConfig(
    context_length=CONTEXT_LENGTH,
    output_patch_size=PATCH_SIZE,
    input_patch_size=PATCH_SIZE,
    input_patch_stride=PATCH_SIZE,
    quantiles=QUANTILES,
    time_encoding_scale=CONTEXT_LENGTH,
    use_reg_token=True,
    use_arcsinh=True,
    max_output_patches=64,
)

model_config = Chronos2CoreConfig(
    d_model=D_MODEL,
    d_kv=D_KV,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout_rate=DROPOUT_RATE,
    vocab_size=VOCAB_SIZE,
)
model_config.chronos_config = chronos_forecasting_config.__dict__
model = Chronos2Model(model_config)

# Trainer
trainer = Chronos2Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print("Starting training...")
trainer.train()

In [ ]:
# Save to Hub
if HF_TOKEN:
    print("Pushing model to Hugging Face Hub...")
    trainer.push_to_hub()
    print("Model pushed successfully.")
else:
    print("HF_TOKEN not found. Skipping push to hub.")

In [ ]:
# Load Model for Evaluation
print("Loading trained model for evaluation...")

# Construct Repo ID
repo_id = f"voyagersnlppolito/{RUN_NAME}"

print(f"Loading model from Hub: {repo_id}")
pipeline = Chronos2Pipeline.from_pretrained(repo_id, token=HF_TOKEN)
pipeline.model.to("cuda" if torch.cuda.is_available() else "cpu")
print("Model loaded successfully.")


In [ ]:
print("Starting Standardized Evaluation...")
RESULTS_PATH = "evaluation_results.csv"
CONFIG_PATH = "scripts/evaluation/configs/in-domain.yaml"

# Ensure config exists
if not os.path.exists(CONFIG_PATH):
    print(f"Config file not found at {CONFIG_PATH}. Please check the path.")
else:
    # Run evaluation
    # We use the pipeline loaded in the previous cell
    eval_pipeline_and_save_results(
        pipeline=pipeline,
        config_path=CONFIG_PATH,
        metrics_path=RESULTS_PATH,
        model_id=RUN_NAME,
        batch_size=BATCH_SIZE,
    )

    # Log to WandB
    if wandb.run is not None:
        print("Logging results to WandB...")
        try:
            results_df = pd.read_csv(RESULTS_PATH)
            wandb.log({"evaluation_results": wandb.Table(dataframe=results_df)})
            
            # Log aggregate metrics
            avg_mase = results_df["MASE"].mean()
            avg_wql = results_df["WQL"].mean()
            wandb.log({"eval/avg_mase": avg_mase, "eval/avg_wql": avg_wql})
            print("Logged results to WandB.")
        except Exception as e:
            print(f"Failed to log to WandB: {e}")
    else:
        print("WandB run not active. Skipping logging.")

    # Push to Hub
    if HF_TOKEN:
        print("Uploading evaluation results to Hub...")
        api = HfApi()
        try:
            api.upload_file(
                path_or_fileobj=RESULTS_PATH,
                path_in_repo="evaluation_results.csv",
                repo_id=f"voyagersnlppolito/{RUN_NAME}", # Infer username
                repo_type="model"
            )
            print("Evaluation results uploaded successfully.")
        except Exception as e:
            print(f"Failed to upload results: {e}")
    else:
        print("HF_TOKEN not found. Skipping upload.")


In [ ]:
# Run Zero-Shot Evaluation
print("Starting Zero-Shot Evaluation...")
ZERO_SHOT_RESULTS_PATH = "evaluation_results_zero_shot.csv"
ZERO_SHOT_CONFIG_PATH = "scripts/evaluation/configs/zero-shot.yaml"

if not os.path.exists(ZERO_SHOT_CONFIG_PATH):
    print(f"Config file not found at {ZERO_SHOT_CONFIG_PATH}. Please check the path.")
else:
    # Run evaluation
    eval_pipeline_and_save_results(
        pipeline=pipeline,
        config_path=ZERO_SHOT_CONFIG_PATH,
        metrics_path=ZERO_SHOT_RESULTS_PATH,
        model_id=RUN_NAME,
        batch_size=BATCH_SIZE,
    )

    # Log to WandB
    if wandb.run is not None:
        print("Logging zero-shot results to WandB...")
        try:
            zs_results_df = pd.read_csv(ZERO_SHOT_RESULTS_PATH)
            wandb.log({"evaluation_results_zero_shot": wandb.Table(dataframe=zs_results_df)})
            
            # Log aggregate metrics
            avg_mase = zs_results_df["MASE"].mean()
            avg_wql = zs_results_df["WQL"].mean()
            wandb.log({"eval/zero_shot_avg_mase": avg_mase, "eval/zero_shot_avg_wql": avg_wql})
            print("Logged zero-shot results to WandB.")
        except Exception as e:
            print(f"Failed to log to WandB: {e}")
    else:
        print("WandB run not active. Skipping logging.")

    # Push to Hub
    if HF_TOKEN:
        print("Uploading zero-shot evaluation results to Hub...")
        api = HfApi()
        try:
            api.upload_file(
                path_or_fileobj=ZERO_SHOT_RESULTS_PATH,
                path_in_repo="evaluation_results_zero_shot.csv",
                repo_id=f"voyagersnlppolito/{RUN_NAME}",
                repo_type="model"
            )
            print("Zero-shot evaluation results uploaded successfully.")
        except Exception as e:
            print(f"Failed to upload results: {e}")
    else:
        print("HF_TOKEN not found. Skipping upload.")
